# importing libraries

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# loading dataset

In [2]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# preprocessing

##  label encoding

In [3]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## splitting

In [4]:
X_train, X_test, y_train, y_test = train = train_test_split(df['text'], df['label_enc'], test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = X_train.to_numpy(), X_test.to_numpy(), y_train.to_numpy(), y_test.to_numpy()


# vocabulary stats

In [5]:
averageWordsLength = round(sum([len(i.split()) for i in df['text']]) / len(df['text']))
totalWordsLength = len(set(" ".join(df['text']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 4457
Average words per message: 16
Approximate vocabulary size: 15686


# model development

In [6]:
cnn = tf.keras.models.Sequential()
cnn.add(tf.keras.layers.Input(shape=(1,), dtype=tf.string))

In [7]:
text_vec = tf.keras.layers.TextVectorization(
    max_tokens=totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=averageWordsLength,
)
text_vec.adapt(X_train)
cnn.add(text_vec)

In [8]:
cnn.add(tf.keras.layers.Embedding(
    input_dim=totalWordsLength,
    output_dim=128,
    embeddings_initializer='uniform',
    input_length=averageWordsLength
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
cnn.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation="relu"))
cnn.add(tf.keras.layers.GlobalAveragePooling1D())
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
# cnn.add(tf.keras.layers.Dropout(0.2))
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [10]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 16)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 16, 128)        │     2,007,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 12, 128)        │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,106,497 (8.04 MB)

 Trainable params: 2,106,497 (8.04 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

# model training

In [12]:
# cnn.fit(x = training_set, validation_data = test_set, epochs = 25)
history = cnn.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), validation_steps=int(0.2 * len(X_test)))

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.8635 - loss: 0.3366 - val_accuracy: 0.9749 - val_loss: 0.0779
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step - accuracy: 0.9861 - loss: 0.0468 - val_accuracy: 0.9776 - val_loss: 0.0771
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.9948 - loss: 0.0168 - val_accuracy: 0.9794 - val_loss: 0.0922
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.9993 - loss: 0.0050 - val_accuracy: 0.9767 - val_loss: 0.1058
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.9995 - loss: 0.0028 - val_accuracy: 0.9758 - val_loss: 0.1314


# model evaluation

In [13]:
test_messages = [
    "Hey, are we meeting today?",
    "WIN cash now!!! Click the link",
    "Your OTP is 348921",
    "You have won $1000!",
    "Congratulations! You have won a free lottery ticket",
    "Don't forget to submit the assignment",
    "URGENT! You won a cash prize. Call immediately!",
    "My friend won a prize yesterday"
]

# Convert the list of messages to a TensorFlow constant of strings
test_messages_tf = tf.constant(test_messages, dtype=tf.string)

preds = cnn.predict(test_messages_tf)

for msg, p in zip(test_messages, preds):
    label = "Spam" if p[0] >= 0.4 else "Ham"
    print(f"{label} → {msg} ({p[0]:.2f})")
    # print(f"{label} → {msg} ({p[0]})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
Ham → Hey, are we meeting today? (0.00)
Spam → WIN cash now!!! Click the link (0.97)
Ham → Your OTP is 348921 (0.11)
Spam → You have won $1000! (0.99)
Spam → Congratulations! You have won a free lottery ticket (1.00)
Ham → Don't forget to submit the assignment (0.00)
Spam → URGENT! You won a cash prize. Call immediately! (1.00)
Spam → My friend won a prize yesterday (1.00)
